In [1]:
import numpy as np
import pandas as pd
import torch
import os
import matplotlib.pyplot as plt

d:\Program Files\anaconda3\envs\torch126\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
import sys

from flare_preprocessing import *
from utilities import *

In [3]:
%load_ext autoreload
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [4]:
opr = opr_data_preprocessing("D:/2024_S1/ML_SEP_2402/swpc_ftp/v2_ftp_flares_1997_2024.csv")
sci = sci_data_preprocessing("D:/2024_S1/ML_SEP_2402/Sci_matched_with_assigned_ar_20100101_20240721.csv", opr)

d:\2024_S1\ML_SEP_2402\Final_update\Open_Repo\scripts_model_training\utilities.py:64: RuntimeWarning: divide by zero encountered in log10
  return np.log10(intensity * 10 ** -8)


Shape of the operation data: (25364, 9)
Shape of the science-quality data: (30958, 31)


In [5]:
from New_SampleConstruction import *
from New_lstm2 import *

In [6]:
from pathlib import Path

def read_all_harp_csvs(root: str | Path, recursive: bool = False) -> pd.DataFrame:
    """
    Read all per-HARP CSVs under `root`, concatenate them in ascending HARP order,
    Assumes files are named like HARP_<harpnum>.csv.
    """

    root = Path(root)
    pattern = "**/HARP_*.csv" if recursive else "HARP_*.csv"

    # List files first, sorted by HARP number
    harp_files = []
    for f in root.glob(pattern):
        try:
            harpnum = int(f.stem.split("_")[1])
            harp_files.append((harpnum, f))
        except Exception:
            continue

    # sort by numeric harp number
    harp_files.sort(key=lambda x: x[0])

    frames = []

    for harpnum, f in harp_files:
        # read with T_REC parsed as datetime
        df = pd.read_csv(f, parse_dates=["T_REC"])
        print(f"Reading HARP {harpnum}")

        # Skip if the HARP lifetime is less than 120 rows (24h / 12min cadence)
        if len(df) < 120:
            print(f"  -> Skipping HARP {harpnum} (too short)")
            continue
        # fill the HARPNUM with the harpnum
        df['HARPNUM'] = harpnum
        frames.append(df)

    if not frames:
        print("No valid HARP CSVs found.")
        return pd.DataFrame()

    # Concatenate in correct order
    full_df = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(frames)} HARPs; total rows = {len(full_df)}")
    
    return full_df

In [7]:
nrt_sharps = read_all_harp_csvs(r"D:\\Input Data\\Operational Data\\HMI\SHARP_by_HARP")

Reading HARP 476
Reading HARP 487
Reading HARP 493
Reading HARP 495
Reading HARP 497
Reading HARP 498
Reading HARP 499
  -> Skipping HARP 499 (too short)
Reading HARP 500
  -> Skipping HARP 500 (too short)
Reading HARP 501
Reading HARP 502
Reading HARP 503
Reading HARP 504
Reading HARP 505
Reading HARP 506
  -> Skipping HARP 506 (too short)
Reading HARP 508
Reading HARP 509
Reading HARP 510
Reading HARP 511
Reading HARP 512
Reading HARP 513
  -> Skipping HARP 513 (too short)
Reading HARP 514
  -> Skipping HARP 514 (too short)
Reading HARP 515
Reading HARP 516
Reading HARP 518
  -> Skipping HARP 518 (too short)
Reading HARP 520
Reading HARP 521
Reading HARP 523
Reading HARP 524
  -> Skipping HARP 524 (too short)
Reading HARP 526
Reading HARP 527
Reading HARP 529
Reading HARP 532
Reading HARP 533
  -> Skipping HARP 533 (too short)
Reading HARP 534
Reading HARP 537
Reading HARP 547
  -> Skipping HARP 547 (too short)
Reading HARP 550
Reading HARP 552
  -> Skipping HARP 552 (too short)
Read

C:\Users\huke0\AppData\Local\Temp\ipykernel_36768\2427577827.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df = pd.read_csv(f, parse_dates=["T_REC"])


Reading HARP 11095
  -> Skipping HARP 11095 (too short)
Reading HARP 11096
  -> Skipping HARP 11096 (too short)
Reading HARP 11097
Reading HARP 11098
Reading HARP 11099
Reading HARP 11100
Reading HARP 11101
Reading HARP 11102
Reading HARP 11103
  -> Skipping HARP 11103 (too short)
Reading HARP 11104
Reading HARP 11105
Reading HARP 11106
  -> Skipping HARP 11106 (too short)
Reading HARP 11107
  -> Skipping HARP 11107 (too short)
Reading HARP 11108
  -> Skipping HARP 11108 (too short)
Reading HARP 11123
  -> Skipping HARP 11123 (too short)
Reading HARP 11172
Reading HARP 11174
Reading HARP 11197
Reading HARP 11199
Reading HARP 11200
Reading HARP 11203
Reading HARP 11207
  -> Skipping HARP 11207 (too short)
Reading HARP 11209
Reading HARP 11210
Reading HARP 11211
  -> Skipping HARP 11211 (too short)
Reading HARP 11212
  -> Skipping HARP 11212 (too short)
Reading HARP 11215
Reading HARP 11217
Reading HARP 11218
Reading HARP 11219
Reading HARP 11220
  -> Skipping HARP 11220 (too short)
Read

In [ ]:
# delete rows with key columns any NaN values
key_columns = [
    'USFLUX','MEANGAM','MEANGBT','MEANGBZ','MEANGBH','MEANJZD',
    'TOTUSJZ','MEANALP','MEANJZH','TOTUSJH','ABSNJZH','SAVNCPP',
    'MEANPOT','TOTPOT','MEANSHR','SHRGT45','SIZE','SIZE_ACR',
    'NACR','NPIX'
]
nrt_sharps = nrt_sharps.dropna(subset=key_columns)

In [41]:
nrt_sharps[nrt_sharps['HARPNUM'] == 911]['HARPNUM']

112907    911
112908    911
112909    911
112910    911
112911    911
         ... 
113653    911
113654    911
113655    911
113656    911
113657    911
Name: HARPNUM, Length: 685, dtype: int64

In [37]:
oneharp = nrt_sharps[nrt_sharps['HARPNUM'] == 911]
oneharp = oneharp.sort_values(by=['T_REC'])
# delete the duplicate rows by 'T_REC', only 3 harps have duplicate rows
oneharp = oneharp.drop_duplicates(subset=['T_REC'], keep='first')

In [38]:
oneharp

,T_REC,HARPNUM,NOAA_AR,LAT_MIN,LON_MIN,LAT_MAX,LON_MAX,USFLUX,MEANGAM,MEANGBT,...,SAVNCPP,MEANPOT,TOTPOT,MEANSHR,SHRGT45,SIZE,SIZE_ACR,NACR,NPIX,QUALITY
112907,2013-01-26 07:36:00,911,0.0,6.954847,-40.411224,8.372826,-38.359570,3.515588e+20,38.442,170.717,...,5.041572e+11,5094.478,3.267829e+21,30.565,16.356,174.001541,16.965364,199.0,2041.0,1.024000e+03
112908,2013-01-26 07:48:00,911,0.0,6.948073,-40.294506,8.396740,-38.242920,3.502688e+20,36.713,187.401,...,2.767699e+11,5425.789,3.365056e+21,31.353,17.345,178.776169,16.027620,188.0,2097.0,6.656000e+04
112909,2013-01-26 08:00:00,911,0.0,6.940814,-40.178429,8.453302,-38.018875,3.761840e+20,35.823,274.295,...,2.526030e+12,8429.553,5.519046e+21,36.319,29.412,193.184967,18.500061,217.0,2266.0,6.656000e+04
112910,2013-01-26 08:12:00,911,0.0,6.777422,-40.351238,8.697717,-37.641171,4.307237e+20,37.866,191.477,...,3.356216e+12,8503.141,5.973757e+21,35.025,24.575,303.077026,30.947105,363.0,3555.0,2.147485e+09
112911,2013-01-26 08:24:00,911,0.0,6.741978,-40.274921,8.723409,-37.526470,4.488171e+20,35.577,201.807,...,4.127659e+12,8397.834,5.966691e+21,33.796,20.561,317.228729,31.202824,366.0,3721.0,1.024000e+03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113653,2013-02-01 12:48:00,911,0.0,7.535683,44.178802,9.660058,47.578308,0.000000e+00,36.989,86.989,...,0.000000e+00,1913.241,0.000000e+00,41.647,0.000,361.281738,21.256887,249.0,4232.0,1.024000e+03
113654,2013-02-01 13:00:00,911,0.0,7.547003,44.306927,9.661375,47.693962,0.000000e+00,36.989,86.989,...,0.000000e+00,1913.241,0.000000e+00,41.647,0.000,354.445618,20.744287,243.0,4152.0,1.024000e+03
113655,2013-02-01 13:12:00,911,0.0,7.578364,44.434383,9.658612,47.795174,0.000000e+00,36.989,86.989,...,0.000000e+00,1913.241,0.000000e+00,41.647,0.000,347.094696,21.341288,250.0,4066.0,1.024000e+03
113656,2013-02-01 13:24:00,911,0.0,7.587598,44.561337,9.650886,47.909428,0.000000e+00,36.989,86.989,...,0.000000e+00,1913.241,0.000000e+00,41.647,0.000,341.025909,21.340796,250.0,3995.0,1.024000e+03


In [42]:
from New_SampleConstruction import *

In [43]:
sci_obj_24 = New_SampleConstruction()
sci_obj_24.samples_from_harp(sci, nrt_sharps, lead_window=0, forecasting_window=24)

Processing HARPNUM: 476
120 rows of data for HARPNUM 476
Processing HARPNUM: 487
268 rows of data for HARPNUM 487
Processing HARPNUM: 493
691 rows of data for HARPNUM 493
Processing HARPNUM: 495
844 rows of data for HARPNUM 495
Processing HARPNUM: 497
722 rows of data for HARPNUM 497
Processing HARPNUM: 498
96 rows of data for HARPNUM 498
Processing HARPNUM: 501
456 rows of data for HARPNUM 501
Processing HARPNUM: 502
105 rows of data for HARPNUM 502
Processing HARPNUM: 503
746 rows of data for HARPNUM 503
Processing HARPNUM: 504
875 rows of data for HARPNUM 504
Processing HARPNUM: 505
436 rows of data for HARPNUM 505
Processing HARPNUM: 508
605 rows of data for HARPNUM 508
Processing HARPNUM: 509
370 rows of data for HARPNUM 509
Processing HARPNUM: 510
538 rows of data for HARPNUM 510
Processing HARPNUM: 511
277 rows of data for HARPNUM 511
Processing HARPNUM: 512
427 rows of data for HARPNUM 512
Processing HARPNUM: 515
482 rows of data for HARPNUM 515
Processing HARPNUM: 516
384 rows

In [44]:
opr_obj_24 = New_SampleConstruction()
opr_obj_24.samples_from_harp(opr, nrt_sharps, lead_window=0, forecasting_window=24)

Processing HARPNUM: 476
120 rows of data for HARPNUM 476
Processing HARPNUM: 487
268 rows of data for HARPNUM 487
Processing HARPNUM: 493
691 rows of data for HARPNUM 493
Processing HARPNUM: 495
844 rows of data for HARPNUM 495
Processing HARPNUM: 497
722 rows of data for HARPNUM 497
Processing HARPNUM: 498
96 rows of data for HARPNUM 498
Processing HARPNUM: 501
456 rows of data for HARPNUM 501
Processing HARPNUM: 502
105 rows of data for HARPNUM 502
Processing HARPNUM: 503
746 rows of data for HARPNUM 503
Processing HARPNUM: 504
875 rows of data for HARPNUM 504
Processing HARPNUM: 505
436 rows of data for HARPNUM 505
Processing HARPNUM: 508
605 rows of data for HARPNUM 508
Processing HARPNUM: 509
370 rows of data for HARPNUM 509
Processing HARPNUM: 510
538 rows of data for HARPNUM 510
Processing HARPNUM: 511
277 rows of data for HARPNUM 511
Processing HARPNUM: 512
427 rows of data for HARPNUM 512
Processing HARPNUM: 515
482 rows of data for HARPNUM 515
Processing HARPNUM: 516
384 rows

In [45]:
sci_obj_12 = New_SampleConstruction()
sci_obj_12.samples_from_harp(sci, nrt_sharps, lead_window=0, forecasting_window=12)

Processing HARPNUM: 476
120 rows of data for HARPNUM 476
Processing HARPNUM: 487
268 rows of data for HARPNUM 487
Processing HARPNUM: 493
691 rows of data for HARPNUM 493
Processing HARPNUM: 495
844 rows of data for HARPNUM 495
Processing HARPNUM: 497
722 rows of data for HARPNUM 497
Processing HARPNUM: 498
96 rows of data for HARPNUM 498
Processing HARPNUM: 501
456 rows of data for HARPNUM 501
Processing HARPNUM: 502
105 rows of data for HARPNUM 502
Processing HARPNUM: 503
746 rows of data for HARPNUM 503
Processing HARPNUM: 504
875 rows of data for HARPNUM 504
Processing HARPNUM: 505
436 rows of data for HARPNUM 505
Processing HARPNUM: 508
605 rows of data for HARPNUM 508
Processing HARPNUM: 509
370 rows of data for HARPNUM 509
Processing HARPNUM: 510
538 rows of data for HARPNUM 510
Processing HARPNUM: 511
277 rows of data for HARPNUM 511
Processing HARPNUM: 512
427 rows of data for HARPNUM 512
Processing HARPNUM: 515
482 rows of data for HARPNUM 515
Processing HARPNUM: 516
384 rows

In [46]:
opr_obj_12 = New_SampleConstruction()
opr_obj_12.samples_from_harp(opr, nrt_sharps, lead_window=0, forecasting_window=12)

Processing HARPNUM: 476
120 rows of data for HARPNUM 476
Processing HARPNUM: 487
268 rows of data for HARPNUM 487
Processing HARPNUM: 493
691 rows of data for HARPNUM 493
Processing HARPNUM: 495
844 rows of data for HARPNUM 495
Processing HARPNUM: 497
722 rows of data for HARPNUM 497
Processing HARPNUM: 498
96 rows of data for HARPNUM 498
Processing HARPNUM: 501
456 rows of data for HARPNUM 501
Processing HARPNUM: 502
105 rows of data for HARPNUM 502
Processing HARPNUM: 503
746 rows of data for HARPNUM 503
Processing HARPNUM: 504
875 rows of data for HARPNUM 504
Processing HARPNUM: 505
436 rows of data for HARPNUM 505
Processing HARPNUM: 508
605 rows of data for HARPNUM 508
Processing HARPNUM: 509
370 rows of data for HARPNUM 509
Processing HARPNUM: 510
538 rows of data for HARPNUM 510
Processing HARPNUM: 511
277 rows of data for HARPNUM 511
Processing HARPNUM: 512
427 rows of data for HARPNUM 512
Processing HARPNUM: 515
482 rows of data for HARPNUM 515
Processing HARPNUM: 516
384 rows

In [47]:
sci_obj_6 = New_SampleConstruction()
sci_obj_6.samples_from_harp(sci, nrt_sharps, lead_window=0, forecasting_window=6)

Processing HARPNUM: 476
120 rows of data for HARPNUM 476
Processing HARPNUM: 487
268 rows of data for HARPNUM 487
Processing HARPNUM: 493
691 rows of data for HARPNUM 493
Processing HARPNUM: 495
844 rows of data for HARPNUM 495
Processing HARPNUM: 497
722 rows of data for HARPNUM 497
Processing HARPNUM: 498
96 rows of data for HARPNUM 498
Processing HARPNUM: 501
456 rows of data for HARPNUM 501
Processing HARPNUM: 502
105 rows of data for HARPNUM 502
Processing HARPNUM: 503
746 rows of data for HARPNUM 503
Processing HARPNUM: 504
875 rows of data for HARPNUM 504
Processing HARPNUM: 505
436 rows of data for HARPNUM 505
Processing HARPNUM: 508
605 rows of data for HARPNUM 508
Processing HARPNUM: 509
370 rows of data for HARPNUM 509
Processing HARPNUM: 510
538 rows of data for HARPNUM 510
Processing HARPNUM: 511
277 rows of data for HARPNUM 511
Processing HARPNUM: 512
427 rows of data for HARPNUM 512
Processing HARPNUM: 515
482 rows of data for HARPNUM 515
Processing HARPNUM: 516
384 rows

In [48]:
opr_obj_6 = New_SampleConstruction()
opr_obj_6.samples_from_harp(opr, nrt_sharps, lead_window=0, forecasting_window=6)

Processing HARPNUM: 476
120 rows of data for HARPNUM 476
Processing HARPNUM: 487
268 rows of data for HARPNUM 487
Processing HARPNUM: 493
691 rows of data for HARPNUM 493
Processing HARPNUM: 495
844 rows of data for HARPNUM 495
Processing HARPNUM: 497
722 rows of data for HARPNUM 497
Processing HARPNUM: 498
96 rows of data for HARPNUM 498
Processing HARPNUM: 501
456 rows of data for HARPNUM 501
Processing HARPNUM: 502
105 rows of data for HARPNUM 502
Processing HARPNUM: 503
746 rows of data for HARPNUM 503
Processing HARPNUM: 504
875 rows of data for HARPNUM 504
Processing HARPNUM: 505
436 rows of data for HARPNUM 505
Processing HARPNUM: 508
605 rows of data for HARPNUM 508
Processing HARPNUM: 509
370 rows of data for HARPNUM 509
Processing HARPNUM: 510
538 rows of data for HARPNUM 510
Processing HARPNUM: 511
277 rows of data for HARPNUM 511
Processing HARPNUM: 512
427 rows of data for HARPNUM 512
Processing HARPNUM: 515
482 rows of data for HARPNUM 515
Processing HARPNUM: 516
384 rows

In [78]:
from New_lstm2 import *

In [81]:
lead0_Mplus2_24_whole = train_lstm(pos_weight=2.0)
lead0_Mplus2_24_whole.train(sci_obj_24.inputs_profile, sci_obj_24.labels, "Mplus2", pd.to_datetime('2010-01-01'), pd.to_datetime('2020-01-01'),n_epoch=15)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (379,) + inhomogeneous part.

In [74]:
time1 = pd.to_datetime('2010-01-01')
time2 = pd.to_datetime('2020-01-01')
obs_point_time = pd.to_datetime(pd.Series([sci_obj_24.inputs_profile[i]['obs_time'] for i in range(len(sci_obj_24.inputs_profile))]))
index = ((obs_point_time >= time1) & (obs_point_time < time2)).values
inputs_profiles = [sci_obj_24.inputs_profile[i] for i in range(len(sci_obj_24.inputs_profile)) if index[i]]
labels = [sci_obj_24.labels[i] for i in range(len(sci_obj_24.labels)) if index[i]]
for i, label in enumerate(labels):
    if not isinstance(label, str):
        print(
            f"Invalid label at index {i}: "
            f"value={repr(label)}, type={type(label)}"
        )

In [53]:
pd.to_datetime(pd.Series([sci_obj_24.inputs_profile[i]['obs_time'] for i in range(len(sci_obj_24.inputs_profile))]))

0      2012-09-15 02:12:00
1      2012-09-15 04:12:00
2      2012-09-23 11:48:00
3      2012-09-23 13:48:00
4      2012-09-23 15:48:00
               ...        
6804   2024-07-02 16:00:00
6805   2024-07-02 18:00:00
6806   2024-07-02 20:00:00
6807   2024-07-02 22:00:00
6808   2024-07-03 00:00:00
Length: 6809, dtype: datetime64[ns]

In [ ]:
working_dir = "D:\\2024_S1\\ML_SEP_2402\\Final_update\\Open_Repo"
save_dir = working_dir + "\\LSTM_models\\WithM_sci_nrtSHARP"

for i, model in enumerate(lead0_Mplus2_24_whole.models):
    os.makedirs(save_dir, exist_ok=True)
    model_path = os.path.join(save_dir, f"lead0_Mplus2_24_whole{i}.pth")
    torch.save(model.state_dict(), model_path)

In [57]:
for label in sci_obj_24.labels:
    print(label)

C1.3
C1.3
C1.3
C1.3
C1.3
C1.3
C1.3
C1.3
C1.3
C1.3
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C2.4
C1.6
C1.6
C1.6
C1.8
C1.8
C1.8
C1.8
C2.0
C2.0
C2.0
C2.0
C2.0
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.9
B4.2
B4.2
B4.2
M1.3
M1.3
M1.3
C1.5
C1.5
C1.5
C1.5
C1.5
C1.5
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C2.1
C1.1
C1.1
C1.1
C1.1
C1.1
C1.1
C1.1
C1.1
C2.0
C2.0
C2.0
C2.0
C2.0
C2.0
C2.0
C2.0
C2.0
C2.0
C2.0
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C4.7
C1.6
C1.6
C1.6
C1.6
C1.9
C1.9
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.8
C2.8
C2.8
C2.8
B8.2
B8.2
B8.2
B9.5
B9.5
B9.5
B9.5
B9.5
B9.5
B9.5
B9.5
B9.5
B9.6
B9.6
B9.6
B9.6
B9.6
B9.6
B9.6
B9.6
B9.6
B9.6
C1.6
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C2.7
C1.8
C1.8
C1.8
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C1.7
C2.8
C2.8
C2.8
C2.8
C2.8
C2.8
C2.8
C2.8
C2.7
C2.7
C2.2
C2.2
C2.2
C2.2
C2.2
C2.2
C2.2
C2.2
C2.2
C2.2
C2.2
C1.0
C1.0
C1.0
C1.0
C1.0
C1.0
